In [1]:
# 05_tightness_monotonicity.ipynb
# Propositions (tightness and monotonicity): the worst-case bound P*Delta/2
# is attained by a half-lattice construction; the number of distinct integer
# scores scales like W/Delta; migration is monotone in Delta.

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from sklearn.metrics import roc_auc_score
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"savefig.dpi": 600, "figure.dpi": 110, "axes.grid": True,
                     "grid.color": "0.9", "font.size": 9, "axes.edgecolor": "0.3"})
GREYS = ["0.0", "0.3", "0.5", "0.65"]; LS = ["-", "--", "-.", ":"]; MK = ["o", "s", "^", "D"]
Path("../results/figures").mkdir(parents=True, exist_ok=True)
Path("../results/tables").mkdir(parents=True, exist_ok=True)
def _save(fig, name):
    fig.savefig(f"../results/figures/{name}.png", bbox_inches="tight")
    fig.savefig(f"../results/figures/{name}.pdf", bbox_inches="tight")

df = pd.read_parquet("../data/synthetic_population.parquet")
ccols = [c for c in df.columns if c.startswith("c")]
C = df[ccols].to_numpy(); f = df["f"].to_numpy(); p = df["p"].to_numpy(); y = df["y"].to_numpy()
P = len(ccols); b0 = float((f - C.sum(1)).mean())
def quant(A, D): return D * np.round(A / D)
def fq(D): return b0 + quant(C, D).sum(1)

Deltas = np.array([0.01, 0.02, 0.05, 0.1, 0.2, 0.4])
W = float(np.ptp(f))

def migration(fc, fqv, K):
    q = np.quantile(fc, np.linspace(0, 1, K + 1)[1:-1])
    return float(np.mean(np.digitize(fc, q) != np.digitize(fqv, q)))

rows = []
for D in Deltas:
    # worst-case cells: residual just below Delta/2 on every active cell
    Cwc = np.full((200, P), D * (0.5 - 1e-6))
    e_wc = (D * np.round(Cwc / D) - Cwc).sum(1)
    nscore = int(len(np.unique(np.round((fq(D) - b0) / D).astype(np.int64))))
    rows.append(dict(Delta=D, worst_bound=P * D / 2, achieved=np.abs(e_wc).max(),
                     distinct_scores=nscore, W_over_Delta=W / D,
                     migration_K10=migration(f, fq(D), 10)))
tab = pd.DataFrame(rows)
tab.to_csv("../results/tables/05_tightness_monotonicity.csv", index=False)
print(tab.round(3).to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
ax[0].plot(tab.Delta, tab.worst_bound, ls="--", color="0.0", marker="s", label=r"bound $P\Delta/2$")
ax[0].plot(tab.Delta, tab.achieved, ls="-", color="0.4", marker="o", label="achieved max|e|")
ax[0].set_xlabel(r"lattice spacing $\Delta$"); ax[0].set_ylabel("worst-case log-odds perturbation")
ax[0].legend(fontsize=7)
ax[1].plot(tab.Delta, tab.distinct_scores, ls="-", color="0.0", marker="o", label="distinct scores")
ax[1].plot(tab.Delta, tab.W_over_Delta, ls=":", color="0.5", marker="^", label=r"$W/\Delta$ reference")
ax[1].set_xscale("log"); ax[1].set_yscale("log")
ax[1].set_xlabel(r"lattice spacing $\Delta$"); ax[1].set_ylabel("number of distinct integer scores")
ax[1].legend(fontsize=7)
fig.tight_layout(); _save(fig, "05_tightness_monotonicity"); plt.close(fig)

 Delta  worst_bound  achieved  distinct_scores  W_over_Delta  migration_K10
  0.01          0.1       0.1              889       900.619          0.015
  0.02          0.2       0.2              446       450.310          0.027
  0.05          0.5       0.5              181       180.124          0.076
  0.10          1.0       1.0               91        90.062          0.140
  0.20          2.0       2.0               51        45.031          0.297
  0.40          4.0       4.0               17        22.515          0.678
